# Aula 17 — Bootstrap e testes de permutação

Laboratório reproduzível para a disciplina **02-statistics** do AI Lab.

Vamos estimar incerteza para estatísticas não lineares, comparar modelos de modo pareado e verificar por que a unidade de reamostragem importa.

**Ambiente:** Python 3.10+; NumPy ≥ 1.24; pandas ≥ 2.0; SciPy ≥ 1.11; Matplotlib ≥ 3.7.  
**Seed global:** `20260907`.


## Protocolo metodológico

1. Definir a estatística e a unidade independente antes de reamostrar.
2. Preservar pares e grupos.
3. Fixar a seed e documentar o número de réplicas.
4. Usar `(b + 1) / (B + 1)` em permutação Monte Carlo.
5. Tratar os modelos como fixos no bootstrap por casos deste laboratório.

As simulações são didáticas: resultados de produção exigem um desenho compatível com a coleta real.


In [ ]:
# Dependências: numpy>=1.24, pandas>=2.0, scipy>=1.11, matplotlib>=3.7
import itertools
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats

SEED = 20260907
rng = np.random.default_rng(SEED)

print(f"Seed: {SEED}")
print(f"NumPy: {np.__version__} | SciPy: {scipy.__version__}")


## 1. Bootstrap da mediana de latência

Latências costumam ser assimétricas. Geramos uma amostra lognormal e tratamos cada requisição como independente apenas para esta demonstração. O alvo é a mediana populacional.


In [ ]:
n = 250
latencias = rng.lognormal(mean=np.log(180), sigma=0.55, size=n)
mediana_observada = float(np.median(latencias))

print(f"n = {n}")
print(f"Mediana observada = {mediana_observada:.3f} ms")
print(f"Média observada = {latencias.mean():.3f} ms")

assert mediana_observada > 0
assert latencias.mean() > mediana_observada  # assimetria à direita nesta amostra


In [ ]:
B_MEDIANA = 20_000
rng_mediana = np.random.default_rng(SEED + 1)
indices = rng_mediana.integers(0, n, size=(B_MEDIANA, n))
medianas_boot = np.median(latencias[indices], axis=1)

ic_percentil = np.quantile(medianas_boot, [0.025, 0.975])
se_boot = float(np.std(medianas_boot, ddof=1))

# Implementação oficial do SciPy para o intervalo BCa.
resultado_bca = stats.bootstrap(
    (latencias,),
    np.median,
    confidence_level=0.95,
    n_resamples=9_999,
    method="BCa",
    rng=np.random.default_rng(SEED + 2),
)
ic_bca = np.array([
    resultado_bca.confidence_interval.low,
    resultado_bca.confidence_interval.high,
])

resumo_mediana = pd.DataFrame({
    "método": ["percentil", "BCa (SciPy)"],
    "limite_inferior_ms": [ic_percentil[0], ic_bca[0]],
    "limite_superior_ms": [ic_percentil[1], ic_bca[1]],
})
print(resumo_mediana.round(3).to_string(index=False))
print(f"Erro-padrão bootstrap = {se_boot:.3f} ms")

assert ic_percentil[0] < mediana_observada < ic_percentil[1]
assert ic_bca[0] < mediana_observada < ic_bca[1]
assert se_boot > 0


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(medianas_boot, bins=45, color="#2563eb", alpha=0.78, edgecolor="white")
ax.axvline(mediana_observada, color="#111827", linewidth=2, label="mediana observada")
ax.axvspan(ic_percentil[0], ic_percentil[1], color="#f59e0b", alpha=0.25, label="IC 95% percentil")
ax.set(title="Distribuição bootstrap da mediana", xlabel="Mediana da réplica (ms)", ylabel="Frequência")
ax.legend()
plt.tight_layout()
plt.show()


### Estabilidade Monte Carlo

O intervalo também varia por causa do número finito de réplicas. Usamos prefixos da mesma sequência para observar a estabilização sem confundir o efeito com seeds diferentes.


In [ ]:
tamanhos_B = [200, 1_000, 5_000, 20_000]
estabilidade = []
for b in tamanhos_B:
    lo, hi = np.quantile(medianas_boot[:b], [0.025, 0.975])
    estabilidade.append({"B": b, "IC_inferior": lo, "IC_superior": hi, "largura": hi - lo})

estabilidade_df = pd.DataFrame(estabilidade)
print(estabilidade_df.round(3).to_string(index=False))
assert estabilidade_df["largura"].gt(0).all()


## 2. Bootstrap pareado da diferença de F1

Os classificadores A e B avaliam os mesmos 1.200 casos. Cada réplica sorteia índices de casos e aplica esses mesmos índices a `y`, `pred_A` e `pred_B`. Assim, preservamos a correlação entre modelos.


In [ ]:
def f1_score_binario(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=bool)
    y_pred = np.asarray(y_pred, dtype=bool)
    tp = np.sum(y_true & y_pred)
    fp = np.sum(~y_true & y_pred)
    fn = np.sum(y_true & ~y_pred)
    denominador = 2 * tp + fp + fn
    return float(2 * tp / denominador) if denominador else np.nan

rng_f1 = np.random.default_rng(SEED + 3)
n_casos = 1_200
y = rng_f1.binomial(1, 0.25, n_casos).astype(bool)
u_sens = rng_f1.random(n_casos)
u_fp = rng_f1.random(n_casos)

# B tem sensibilidade e especificidade ligeiramente melhores, usando choques compartilhados.
pred_a = np.where(y, u_sens < 0.72, u_fp < 0.10)
pred_b = np.where(y, u_sens < 0.78, u_fp < 0.09)

f1_a = f1_score_binario(y, pred_a)
f1_b = f1_score_binario(y, pred_b)
delta_f1 = f1_b - f1_a

print(f"F1(A) = {f1_a:.6f}")
print(f"F1(B) = {f1_b:.6f}")
print(f"Diferença B - A = {delta_f1:.6f}")
assert 0 <= f1_a <= 1 and 0 <= f1_b <= 1
assert delta_f1 > 0


In [ ]:
B_F1 = 5_000
rng_f1_boot = np.random.default_rng(SEED + 4)
deltas_f1 = np.empty(B_F1)

for b in range(B_F1):
    idx = rng_f1_boot.integers(0, n_casos, n_casos)
    deltas_f1[b] = f1_score_binario(y[idx], pred_b[idx]) - f1_score_binario(y[idx], pred_a[idx])

ic_delta_f1 = np.quantile(deltas_f1, [0.025, 0.975])
print(f"IC 95% percentil para F1(B)-F1(A): [{ic_delta_f1[0]:.6f}; {ic_delta_f1[1]:.6f}]")
print(f"Erro-padrão bootstrap: {np.std(deltas_f1, ddof=1):.6f}")

assert np.isfinite(deltas_f1).all()
assert ic_delta_f1[0] > 0


## 3. Teste de permutação pareado

Agora comparamos perdas contínuas nos mesmos itens. Sob a hipótese nula pareada, inverter aleatoriamente o sinal de cada diferença é equivalente a trocar A e B dentro do item.

O teste é bilateral. Para uma amostra aleatória de permutações, usamos `(extremos + 1) / (B + 1)`.


In [ ]:
rng_perm_dados = np.random.default_rng(SEED + 5)
n_pares = 200
diferencas = rng_perm_dados.normal(loc=0.03, scale=0.15, size=n_pares)
t_observado = float(np.mean(diferencas))

B_PERM = 19_999
rng_perm = np.random.default_rng(SEED + 6)
extremos = 0
estatisticas_perm = np.empty(B_PERM)

# Processamento em lotes evita alocar uma matriz B x n muito grande.
tamanho_lote = 1_000
inicio = 0
while inicio < B_PERM:
    fim = min(inicio + tamanho_lote, B_PERM)
    sinais = rng_perm.choice((-1.0, 1.0), size=(fim - inicio, n_pares))
    valores = np.mean(sinais * diferencas, axis=1)
    estatisticas_perm[inicio:fim] = valores
    extremos += int(np.count_nonzero(np.abs(valores) >= abs(t_observado)))
    inicio = fim

p_monte_carlo = (extremos + 1) / (B_PERM + 1)
p_ingenuo = extremos / B_PERM

print(f"Diferença média observada = {t_observado:.6f}")
print(f"Permutações extremas = {extremos} de {B_PERM}")
print(f"p corrigido = {p_monte_carlo:.6f}")
print(f"p ingênuo = {p_ingenuo:.6f}")
print(f"Menor p possível neste desenho Monte Carlo = {1/(B_PERM+1):.6f}")

assert 0 < p_monte_carlo <= 1
assert math.isclose(p_monte_carlo, (extremos + 1) / (B_PERM + 1))
assert abs(np.mean(estatisticas_perm)) < 0.003


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(estatisticas_perm, bins=45, color="#0f766e", alpha=0.8, edgecolor="white")
ax.axvline(t_observado, color="#dc2626", linewidth=2, label="estatística observada")
ax.axvline(-t_observado, color="#dc2626", linewidth=2, linestyle="--")
ax.set(title="Distribuição nula por permutação pareada", xlabel="Diferença média com sinais permutados", ylabel="Frequência")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Permutação exata

Com poucos pares podemos enumerar todas as combinações de sinais. Aqui há `2**10 = 1.024` combinações, portanto o p-value é calculado diretamente no espaço completo.


In [ ]:
d_pequeno = np.array([0.08, 0.03, -0.01, 0.05, 0.02, 0.07, 0.04, -0.02, 0.06, 0.01])
t_pequeno = float(d_pequeno.mean())
sinais_exatos = np.array(list(itertools.product((-1.0, 1.0), repeat=len(d_pequeno))))
t_exatos = np.mean(sinais_exatos * d_pequeno, axis=1)
p_exato = float(np.mean(np.abs(t_exatos) >= abs(t_pequeno) - 1e-15))

print(f"Número de permutações exatas = {len(t_exatos)}")
print(f"Diferença observada = {t_pequeno:.6f}")
print(f"p bilateral exato = {p_exato:.6f}")

assert len(t_exatos) == 2 ** len(d_pequeno)
assert 0 < p_exato <= 1


## 5. A unidade muda a incerteza: linhas versus usuários

Geramos dez observações para cada um de 80 usuários. As linhas do mesmo usuário compartilham um efeito aleatório. Comparamos:

- bootstrap ingênuo de 800 linhas;
- bootstrap correto dos 80 usuários, levando as dez linhas de cada usuário.

O estimando é a média por observação em uma população de usuários com dez medições por usuário.


In [ ]:
rng_cluster = np.random.default_rng(SEED + 7)
n_usuarios = 80
linhas_por_usuario = 10
efeito_usuario = rng_cluster.normal(loc=0.02, scale=0.12, size=n_usuarios)
ruido_linha = rng_cluster.normal(loc=0.0, scale=0.04, size=(n_usuarios, linhas_por_usuario))
efeitos = efeito_usuario[:, None] + ruido_linha
efeitos_flat = efeitos.ravel()

print(f"Média observada = {efeitos_flat.mean():.6f}")
print(f"Linhas = {efeitos_flat.size}; usuários independentes = {n_usuarios}")
assert efeitos.shape == (80, 10)


In [ ]:
B_CLUSTER = 5_000
rng_linha = np.random.default_rng(SEED + 8)
rng_usuario = np.random.default_rng(SEED + 9)

idx_linhas = rng_linha.integers(0, efeitos_flat.size, size=(B_CLUSTER, efeitos_flat.size))
medias_linhas = efeitos_flat[idx_linhas].mean(axis=1)

idx_usuarios = rng_usuario.integers(0, n_usuarios, size=(B_CLUSTER, n_usuarios))
# Cada usuário selecionado leva consigo suas dez linhas.
medias_usuarios = efeitos[idx_usuarios].mean(axis=(1, 2))

se_linhas = float(np.std(medias_linhas, ddof=1))
se_usuarios = float(np.std(medias_usuarios, ddof=1))
ic_linhas = np.quantile(medias_linhas, [0.025, 0.975])
ic_usuarios = np.quantile(medias_usuarios, [0.025, 0.975])

comparacao = pd.DataFrame({
    "método": ["linhas (ingênuo)", "usuários (cluster)"],
    "erro_padrão": [se_linhas, se_usuarios],
    "IC_inferior": [ic_linhas[0], ic_usuarios[0]],
    "IC_superior": [ic_linhas[1], ic_usuarios[1]],
})
print(comparacao.round(6).to_string(index=False))
print(f"Razão SE cluster / SE ingênuo = {se_usuarios / se_linhas:.3f}")

assert se_usuarios > 2 * se_linhas
assert (ic_usuarios[1] - ic_usuarios[0]) > (ic_linhas[1] - ic_linhas[0])


## 6. Checagens finais

As asserções abaixo consolidam os pontos metodológicos do laboratório. Se todas passarem, os resultados principais são internamente consistentes e reproduzíveis com a seed informada.


In [ ]:
assert len(medianas_boot) == B_MEDIANA
assert len(deltas_f1) == B_F1
assert len(estatisticas_perm) == B_PERM
assert p_monte_carlo >= 1 / (B_PERM + 1)
assert ic_delta_f1[0] < delta_f1 < ic_delta_f1[1]
assert np.isfinite([mediana_observada, se_boot, delta_f1, p_monte_carlo, se_usuarios]).all()

print("Todas as verificações foram aprovadas.")
print(f"Mediana: {mediana_observada:.3f} ms | IC percentil: [{ic_percentil[0]:.3f}; {ic_percentil[1]:.3f}]")
print(f"ΔF1: {delta_f1:.6f} | IC: [{ic_delta_f1[0]:.6f}; {ic_delta_f1[1]:.6f}]")
print(f"Permutação: p = {p_monte_carlo:.6f} | p exato (amostra pequena) = {p_exato:.6f}")
print(f"SE linha = {se_linhas:.6f} | SE cluster = {se_usuarios:.6f}")


## Desafios

1. Troque a mediana pelo quantil 0,95 e compare os ICs percentil e BCa.
2. Refaça a permutação com `B = 999`, `9.999` e `99.999`; observe a resolução do p-value.
3. Aumente a variância entre usuários e veja como cresce a diferença entre os dois erros-padrão.
4. Implemente a troca pareada de previsões A/B e use a diferença de F1 como estatística de permutação.

Ao adaptar este notebook, documente a unidade independente e a hipótese que torna as permutações válidas.
